In [18]:
from pathlib import Path

BASE_DIR = Path.cwd().parent
EVAL_PATH = BASE_DIR / "receipts" / "eval"

EVAL_IMG_PATH = EVAL_PATH / "img"
EVAL_LABEL_PATH = EVAL_PATH / "entities"

images = list(EVAL_IMG_PATH.glob("*.jpg"))
print(len(images))

146


In [6]:
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification

MODEL_PATH = BASE_DIR / "artifacts" / "receipt_model"

model = LayoutLMv3ForTokenClassification.from_pretrained(MODEL_PATH)

In [9]:
processor = LayoutLMv3Processor.from_pretrained(MODEL_PATH)

In [29]:
import sqlite3

# create / connect
conn = sqlite3.connect("receipts.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS receipts")

# create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS receipts (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    company TEXT,
    date TEXT,
    total REAL,
    address TEXT,
    confidence_json TEXT,
    raw_json TEXT
)
""")

conn.commit()

In [30]:
cursor.execute("PRAGMA table_info(receipts)")
print(cursor.fetchall())

[(0, 'id', 'INTEGER', 0, None, 1), (1, 'company', 'TEXT', 0, None, 0), (2, 'date', 'TEXT', 0, None, 0), (3, 'total', 'REAL', 0, None, 0), (4, 'address', 'TEXT', 0, None, 0), (5, 'confidence_json', 'TEXT', 0, None, 0), (6, 'raw_json', 'TEXT', 0, None, 0)]


In [32]:
import pandas as pd

df = pd.read_sql("SELECT * FROM receipts", conn)
df.head()

,id,company,date,total,address,confidence_json,raw_json


In [33]:
import re

def clean_total(value):
    if not value:
        return None
    
    value = re.sub(r"[^\d.]", "", value)  # keep digits + dot
    try:
        return float(value)
    except:
        return None


def clean_date(value):
    if not value:
        return None
    
    # keep common formats
    match = re.search(r"\d{2}/\d{2}/\d{4}", value)
    return match.group(0) if match else value


def clean_address(value):
    if not value:
        return None
    
    # remove trailing noise like "Document"
    value = re.sub(r"Document.*", "", value, flags=re.IGNORECASE)
    return value.strip()


def clean_company(value):
    if not value:
        return None
    
    return value.strip()

In [34]:
import json

def save_to_db(fields, confidences):
    company = clean_company(fields.get("COMPANY"))
    date = clean_date(fields.get("DATE"))
    total = clean_total(fields.get("TOTAL"))
    address = clean_address(fields.get("ADDRESS"))

    cursor.execute("""
    INSERT INTO receipts (company, date, total, address, confidence_json, raw_json)
    VALUES (?, ?, ?, ?, ?, ?)
    """, (
        company,
        date,
        total,
        address,
        json.dumps(confidences),
        json.dumps(fields)
    ))

    conn.commit()

In [24]:
label_list = [
    "O",
    "B-COMPANY", "I-COMPANY",
    "B-DATE", "I-DATE",
    "B-TOTAL", "I-TOTAL",
    "B-ADDRESS", "I-ADDRESS"
]

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

## Post-processing images
### Converting token predictions to structured fields

In [25]:
import pytesseract

def extract_words_boxes(image):
    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)

    words = []
    boxes = []

    h, w = image.size[1], image.size[0]

    for i, text in enumerate(data["text"]):
        text = text.strip()
        if not text:
            continue

        conf = int(data["conf"][i])
        if conf < 40:
            continue

        x, y, bw, bh = (
            data["left"][i],
            data["top"][i],
            data["width"][i],
            data["height"][i]
        )

        # normalize 0–1000
        box = [
            int(1000 * x / w),
            int(1000 * y / h),
            int(1000 * (x + bw) / w),
            int(1000 * (y + bh) / h),
        ]

        words.append(text)
        boxes.append(box)

    return words, boxes

In [26]:
def extract_fields_with_confidence(tokens, labels, scores):
    results = {}
    confidences = {}

    current_field = None
    buffer = []
    buffer_scores = []

    for token, label, score in zip(tokens, labels, scores):
        if label.startswith("B-"):
            if current_field:
                results[current_field] = " ".join(buffer)
                confidences[current_field] = sum(buffer_scores) / len(buffer_scores)

            current_field = label[2:]
            buffer = [token]
            buffer_scores = [score]

        elif label.startswith("I-") and current_field:
            buffer.append(token)
            buffer_scores.append(score)

        else:
            if current_field:
                results[current_field] = " ".join(buffer)
                confidences[current_field] = sum(buffer_scores) / len(buffer_scores)

                current_field = None
                buffer = []
                buffer_scores = []

    if current_field:
        results[current_field] = " ".join(buffer)
        confidences[current_field] = sum(buffer_scores) / len(buffer_scores)

    return results, confidences

In [27]:
from PIL import Image
import torch.nn.functional as F

def process_receipt(image_path):
    image = Image.open(image_path).convert("RGB")

    # OCR + boxes
    words, boxes = extract_words_boxes(image)

    # Encode
    encoding = processor(
        image,
        words,
        boxes=boxes,
        return_tensors="pt",
        padding="max_length",
        truncation=True
    )

    # Predict
    outputs = model(**encoding)
    probs = F.softmax(outputs.logits, dim=-1)
    preds = probs.argmax(-1)[0]

    # Align tokens
    word_ids = encoding.word_ids(batch_index=0)
    
    final_tokens = []
    final_labels = []
    final_scores = []
    
    prev_word_id = None
    
    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
    
        if word_id != prev_word_id:
            label_id = preds[idx].item()
            score = probs[0, idx, label_id].item()
    
            final_tokens.append(words[word_id])
            final_labels.append(id2label[label_id])
            final_scores.append(score)
    
        prev_word_id = word_id

    # Extract fields
    fields, confidences = extract_fields_with_confidence(final_tokens, final_labels, final_scores)

    # Save
    save_to_db(fields, confidences)

    return fields

In [41]:
results = []

for img_path in images:
    fields = process_receipt(img_path)
    results.append(fields)

results

C:\ProgramData\anaconda3\Lib\site-packages\transformers\modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


[{'COMPANY': 'HOTOR ( SUNGAT RENGIT ) SON, BHD.',
  'ADDRESS': 'Jalan Kurau, Sungai Rengit, 81620 Pengerang, Johor. el;',
  'TOTAL': '20.00',
  'DATE': '23-01-2019'},
 {'COMPANY': 'PERNIAGAAN ZHENG',
  'ADDRESS': 'JALAN PERMAS 9/6 BANDAR BARU PERMAS JAYA 81780 JOHOR TEL',
  'DATE': '09/02/2078',
  'TOTAL': '436.20'},
 {'COMPANY': 'CROSS CHANNEL SDN.',
  'ADDRESS': 'MERANTI 1, SEK. 3, BANDAR UTAMA BATANG KALI, 44300 BATANG KALI, SELANGOR. Tel :',
  'TOTAL': '7.95',
  'DATE': '31/12/2017'},
 {'COMPANY': 'SWC ENTERPRISE SDN BHD',
  'ADDRESS': 'Jalan Mahagoni 7/1, Sekysen 4, Bandar Utama, 44300 Batang Kali, Selangor. Tel :',
  'DATE': '06/01/2018',
  'TOTAL': '8.00'},
 {'COMPANY': 'HORE MASTER HARDWARE & ELECTRICAL',
  'ADDRESS': '& 115G, JALAN SETIA GEMEILANG SETIA ALAM, 40770 BANDAR SETIA ALAM, SELANGOR,',
  'DATE': '22/12/2017',
  'TOTAL': '15.90'},
 {},
 {'ADDRESS': "NO.23 BATU 10, TAMAN SE JALAN KAPAR, 42200 KLANG, S'",
  'TOTAL': '32.70'},
 {'COMPANY': 'LIGHTROOM GALLERY, SDM',
  'AD

In [42]:
df = pd.read_sql("SELECT * FROM receipts", conn)

df.head()

,id,company,date,total,address,confidence_json,raw_json
0,1,"HOTOR ( SUNGAT RENGIT ) SON, BHD.",23-01-2019,20.00,"Jalan Kurau, Sungai Rengit, 81620 Pengerang, J...","{""COMPANY"": 0.9355188267571586, ""ADDRESS"": 0.9...","{""COMPANY"": ""HOTOR ( SUNGAT RENGIT ) SON, BHD...."
1,2,PERNIAGAAN ZHENG,09/02/2078,436.20,JALAN PERMAS 9/6 BANDAR BARU PERMAS JAYA 81780...,"{""COMPANY"": 0.9209394752979279, ""ADDRESS"": 0.9...","{""COMPANY"": ""PERNIAGAAN ZHENG"", ""ADDRESS"": ""JA..."
2,3,CROSS CHANNEL SDN.,31/12/2017,7.95,"MERANTI 1, SEK. 3, BANDAR UTAMA BATANG KALI, 4...","{""COMPANY"": 0.9958662788073221, ""ADDRESS"": 0.9...","{""COMPANY"": ""CROSS CHANNEL SDN."", ""ADDRESS"": ""..."
3,4,SWC ENTERPRISE SDN BHD,06/01/2018,8.00,"Jalan Mahagoni 7/1, Sekysen 4, Bandar Utama, 4...","{""COMPANY"": 0.9968835711479187, ""ADDRESS"": 0.9...","{""COMPANY"": ""SWC ENTERPRISE SDN BHD"", ""ADDRESS..."
4,5,HORE MASTER HARDWARE & ELECTRICAL,22/12/2017,15.90,"& 115G, JALAN SETIA GEMEILANG SETIA ALAM, 4077...","{""COMPANY"": 0.9948287963867187, ""ADDRESS"": 0.9...","{""COMPANY"": ""HORE MASTER HARDWARE & ELECTRICAL..."


In [44]:
EXTRACTED_RECEIPTS = BASE_DIR / "artifacts" / "extracted_receipts.csv"

df.to_csv(EXTRACTED_RECEIPTS, index=False)

In [45]:
import json

conf_df = df["confidence_json"].apply(json.loads).apply(pd.Series)

df_combined = pd.concat([df.drop(columns=["confidence_json"]), conf_df], axis=1)

df_combined.head()

,id,company,date,total,address,raw_json,COMPANY,ADDRESS,TOTAL,DATE
0,1,"HOTOR ( SUNGAT RENGIT ) SON, BHD.",23-01-2019,20.00,"Jalan Kurau, Sungai Rengit, 81620 Pengerang, J...","{""COMPANY"": ""HOTOR ( SUNGAT RENGIT ) SON, BHD....",0.935519,0.974729,0.916327,0.998742
1,2,PERNIAGAAN ZHENG,09/02/2078,436.20,JALAN PERMAS 9/6 BANDAR BARU PERMAS JAYA 81780...,"{""COMPANY"": ""PERNIAGAAN ZHENG"", ""ADDRESS"": ""JA...",0.920939,0.936342,0.997944,0.994884
2,3,CROSS CHANNEL SDN.,31/12/2017,7.95,"MERANTI 1, SEK. 3, BANDAR UTAMA BATANG KALI, 4...","{""COMPANY"": ""CROSS CHANNEL SDN."", ""ADDRESS"": ""...",0.995866,0.977559,0.995155,0.994662
3,4,SWC ENTERPRISE SDN BHD,06/01/2018,8.00,"Jalan Mahagoni 7/1, Sekysen 4, Bandar Utama, 4...","{""COMPANY"": ""SWC ENTERPRISE SDN BHD"", ""ADDRESS...",0.996884,0.964374,0.997133,0.998632
4,5,HORE MASTER HARDWARE & ELECTRICAL,22/12/2017,15.90,"& 115G, JALAN SETIA GEMEILANG SETIA ALAM, 4077...","{""COMPANY"": ""HORE MASTER HARDWARE & ELECTRICAL...",0.994829,0.910657,0.996852,0.998175


In [46]:
df.groupby("company")["total"].sum().sort_values(ascending=False)

company
PERNIAGAAN ZHENG                   872.40
SYARIKAT PERNIAGAAN GIN KEE        639.07
PINGHWAI TRADING SDN BHD           616.59
KEDAI PAPAN YEW CHUAN              374.71
LEE T SON BHD                      374.20
                                    ...  
KHIAM AIK CHAN SDN BHD               0.00
SECRET RECIPE RESTAURANT             0.00
SATU KAMPUNG ENTERPRISE SUN BHD      0.00
Pu Tien                              0.00
SUN VICTORY SON BHD                  0.00
Name: total, Length: 80, dtype: float64

In [47]:
df["total"].sum()

np.float64(8428.329)